# E14a: the frozen main models on internal scans given MCT-LTDiag-like acquisition

**Revision plan, 00 log section 60 (written 23 Oct 2026, before this run).** A blind review asked whether the external failure
comes from acquisition (in MCT-LTDiag every scan has 5 mm slices) or from the population. This notebook gives the internal
scans MCT-LTDiag-like slice density and lets the **frozen** main models predict again. Nothing is trained, tuned or selected;
**the main models and all main results stay unchanged, whatever this shows.**

- **Thick slices by slab averaging:** each new slice is the overlap-weighted mean of the original slices it covers, as in a
  thick reconstruction; masks are averaged the same way, then > 0.5. A scan is never made thinner (the smoothing controls go
  back to the original slice count by linear interpolation, as in E10).
- **Conditions:** unchanged (must reproduce the saved results); **`mct`**: every phase slab-averaged to the MCT-LTDiag slice
  density **R = 0.11698** (E10b's estimate, slices containing liver ÷ largest in-plane liver width; median of the 203
  MCT-LTDiag patients, E12 file), wherever the scan is denser (factor < 1); **`mct_smooth`**: the same, then back to the
  original slice count (thick-slice blur without the change in slice count); **`inject_slab`** (HCC, ICC): the non-contrast
  scan slab-averaged to 0.29 × the portal-venous slice count, with its control **`inject_slab_smooth`**.
- **OA (primary):** held-out HCC-versus-ICC AUC of CNN + radiomics (39 patients) with `mct` inputs against unchanged inputs,
  paired bootstrap (2,000 resamples, seed 42). **Rule fixed in the plan** (internal 0.947, external 0.578, midpoint 0.763):
  "reproduces most of the external loss" if the AUC is < 0.763 and the paired CI excludes 0; "explains part of it" if the CI
  excludes 0 and the AUC is ≥ 0.763; "does not explain it" if the CI includes 0.
- **Secondary:** all three models, `mct_smooth`, out-of-fold (154), three-class macro-AUC, and predicted classes per true
  class (in MCT-LTDiag many HCC and ICC patients were called cHCC-CCA).
- **OC:** E10's rules for `inject_slab` against `inject_slab_smooth` (shift of p(cHCC-CCA) ≥ 0.05, bootstrap CI excluding 0,
  control < 0.05).
- **Gates (the notebook stops if one fails):** unchanged crops equal the Week 2 crops; liver slices and widths equal E10b;
  the frozen CNN and CNN + radiomics reproduce the saved predictions; R equals the value fixed in the plan.

**Runtime:** GPU runtime with High-RAM (L4 is enough), about 1.5–2 hours (reading all 1,112 internal scans from Drive, slab
averaging, radiomics on the CPU cores). **When it finishes,** the last cell saves everything to Drive and disconnects the
runtime (`SHUTDOWN_WHEN_DONE = False` keeps it). **If a cell stops with an error,** the error is saved to Drive
(`E14a_error.txt`) and the runtime disconnects (`SHUTDOWN_ON_ERROR = False` in Cell 1 keeps it). **Safety:** inputs are only
read; outputs go to a new folder `liver_project/revision_2026-10/E14a_<time>/`.

In [ ]:
# Cell 1 — Setup (E10's cell): installs, paths, input checks, output folder E14a_<time>, disconnect on error
import os, sys, json, time, hashlib, platform, datetime, warnings, subprocess, tarfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e14a_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm==4.6.0', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E14A_TEST_ROOT'])
    WORK = Path(os.environ['E14A_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import joblib
import nibabel as nib
import SimpleITK as sitk
import lightgbm as lgb
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
E8B_DIR = REVISION_DIR / 'E8b_20261018T025705Z'
E10_DIR = REVISION_DIR / 'E10_20261019T031225Z' if IN_COLAB else Path(os.environ['E14A_TEST_E10'])
E12_DIR = REVISION_DIR / 'E12_20261022T025246Z' if IN_COLAB else Path(os.environ['E14A_TEST_E12'])
SEED, N_BOOT = 42, 2000
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
CLASSES = ['HCC', 'ICC', 'CHCC']
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
CROP = 96
# fixed in the plan (00 log section 60): MCT-LTDiag slice-density target; E10's non-contrast ratio 0.29 and minimum shift 0.05
R_PLAN = 0.1169811320754717
INJECT_RATIO = 0.29
MIN_SHIFT = 0.05
INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e2_radiomics_oof': (E2_DIR / 'E2_radiomics_noshape_oof_predictions.csv', '21057b42933293a7d45cbefbb6f58dfe89458064a42fff689abcbc826fe28cb6'),
    'e2_radiomics_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e8b_predictions': (E8B_DIR / 'E8b_E9_predictions.csv', '644a520c0d034ce233bb88ab98e1b6d9e89993a661e380d06985edde4aa34064'),
    'e10b_anatomy': (E10_DIR / 'E10b_anatomy_per_scan.csv', 'f7ea15344beba14e25b85aa8fd6879e586670e6d311207c540d45f301ee4183c'),
    'e12_mct_geometry': (E12_DIR / 'E12_mct_crop_geometry.csv', '567bcc76a0eb6e89b6e1c866c793c857244beb1d915ded8afb814c7d96403f70'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {**{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
               **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)}}
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CPU = os.cpu_count() or 2
N_JOBS = max(1, min(8, N_CPU - 1)) if IN_COLAB else 1
N_IO = 3 if IN_COLAB else 2
import sklearn, monai, radiomics
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'lightgbm': lgb.__version__,
            'pyradiomics': radiomics.__version__, 'nibabel': nib.__version__, 'SimpleITK': sitk.Version_VersionString(),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu',
            'cpus': N_CPU}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E14a_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)

SHUTDOWN_ON_ERROR = True   # if any cell stops with an error: save it to Drive (E14a_error.txt) and disconnect the runtime
if IN_COLAB and SHUTDOWN_ON_ERROR:
    from IPython import get_ipython
    def _disconnect_on_error(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        try:
            (OUT_DIR / 'E14a_error.txt').write_text(f'{etype.__name__}: {evalue}')
            from google.colab import drive as _drive, runtime as _runtime
            print('\nThe run stopped with the error above; it is saved in Drive (E14a_error.txt). The runtime disconnects '
                  'in 60 seconds to save compute units; this output stays visible. Send it for review.')
            _drive.flush_and_unmount()
            time.sleep(60)
            _runtime.unassign()
        except Exception as error:
            print('Could not disconnect automatically:', error)
    get_ipython().set_custom_exc((Exception,), _disconnect_on_error)

# the MCT-LTDiag slice-density target R: median of E10b's estimate over the 203 MCT-LTDiag patients (E12 file, primary geometry)
_geo = pd.read_csv(INPUTS['e12_mct_geometry'][0])
R_MCT = float(_geo.loc[_geo['variant'] == 'equalized', 'ratio_before'].median())
print('MCT-LTDiag slice-density target R:', round(R_MCT, 6), '| patients:', int((_geo['variant'] == 'equalized').sum()))
if IN_COLAB and abs(R_MCT - R_PLAN) > 1e-12:
    raise RuntimeError('STOP — the target differs from the value fixed in the plan (0.116981). Send this output.')
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES), '| versions:', VERSIONS, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Data: split, folds, labels, the Week 2 processed crops (for the reproduction gate), patient table
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E14A_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
ALL_IDS = dev_ids + test_ids
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
FOLD_OF = {str(p): f for f, s in enumerate(fold_splits) for p in s['val_ids']}
assert set(FOLD_OF) == set(dev_ids), 'fold files do not cover the development set exactly'
internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
LABEL = internal.set_index('patient_id')['label'].astype(int).to_dict()
CLS = {p: CLASSES[LABEL[p]] for p in ALL_IDS}
table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
table = table[table['patient_id'].isin(ALL_IDS)].drop_duplicates(['patient_id', 'phase'])          # Week 2: first row per phase
ROWS = {pid: g.set_index('phase') for pid, g in table.groupby('patient_id')}
missing = [p for p in ALL_IDS if p not in ROWS or any(ph not in ROWS[p].index for ph in PHASE_ORDER)]
if missing:
    raise RuntimeError(f'STOP — patients without all four phases: {missing[:10]}')
nums = pd.Series({p: int(''.join(ch for ch in p if ch.isdigit())) for p in ALL_IDS})
BLOCKS = {c: (int(nums[[p for p in ALL_IDS if CLS[p] == c]].min()), int(nums[[p for p in ALL_IDS if CLS[p] == c]].max())) for c in CLASSES}
print('Patients:', len(ALL_IDS), '| classes', pd.Series(CLS).value_counts().to_dict(), '| patient-number blocks by class:', BLOCKS)

In [ ]:
# Cell 3 — Internal crops for every condition in one pass over the original scans (E10's Week 2 rebuild; E14a conditions)
def load_nii(path):                          # Week 2, unchanged (returns the array exactly as Week 2 used it)
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def week2_crop(ct, tumor, liver):             # Week 2 process_patient, one phase, unchanged (E10)
    ct_masked = ct * (liver > 0).astype(np.float32)
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    return np.clip(crop_cube(ct_masked, center, CROP), 0, 255).astype(np.uint8), (crop_cube(tumor, center, CROP) > 0)

def resample_slices(vol, n_new, is_mask):     # E10, unchanged; here only used to go back to the original slice count
    """Change only the number of slices (axis 2). CT linear; masks linear then > 0.5."""
    n_new = max(2, int(round(n_new)))
    if n_new == vol.shape[2]:
        return vol
    out = ndimage.zoom(vol.astype(np.float32), (1.0, 1.0, n_new / vol.shape[2]), order=1)
    assert out.shape[2] == n_new, (out.shape, n_new)
    return (out > 0.5).astype(np.float32) if is_mask else out

def slab_average(vol, n_new, is_mask):        # E14a (00 log section 60): thicker slices only, as a thick reconstruction
    """Fewer slices along axis 2: each new slice is the overlap-weighted mean of the original slices it covers. Masks are
    averaged the same way, then > 0.5. A scan is never made thinner: if n_new is not smaller, the volume is returned as it is."""
    n0 = vol.shape[2]
    n_new = max(2, int(round(n_new)))
    if n_new >= n0:
        return vol
    edges = np.arange(n_new + 1) * (n0 / n_new)
    weights = np.zeros((n0, n_new), dtype=np.float64)
    for k in range(n_new):
        for i in range(int(np.floor(edges[k])), min(n0, int(np.ceil(edges[k + 1])))):
            weights[i, k] = max(0.0, min(edges[k + 1], i + 1) - max(edges[k], i))
    weights /= weights.sum(axis=0, keepdims=True)
    out = np.tensordot(vol.astype(np.float32), weights.astype(np.float32), axes=([2], [0]))
    assert out.shape[2] == n_new, (out.shape, n_new)
    return (out > 0.5).astype(np.float32) if is_mask else out.astype(np.float32)

def thicken(phase_vols, n_new, back=False):
    ct, tm, lv = phase_vols
    n0 = ct.shape[2]
    out = [slab_average(ct, n_new, False), slab_average(tm, n_new, True), slab_average(lv, n_new, True)]
    if back:
        out = [resample_slices(out[0], n0, False), resample_slices(out[1], n0, True), resample_slices(out[2], n0, True)]
    return out

def extent(mask, axis):                       # E10, unchanged
    idx = np.where(mask.any(axis=tuple(a for a in range(3) if a != axis)))[0]
    return (int(idx.min()), int(idx.max())) if idx.size else (-1, -1)

def density_ratio(liver):                     # E10b / E12: slices containing liver / largest in-plane liver extent (pixels)
    lv = liver > 0
    n_liver = int(lv.any(axis=(0, 1)).sum())
    if n_liver == 0:
        return float('nan'), 0, 0
    li, lj = extent(lv, 0), extent(lv, 1)
    width = max(li[1] - li[0] + 1, lj[1] - lj[0] + 1)
    return n_liver / max(width, 1), n_liver, width

def process_patient(pid):
    rows = ROWS[pid]
    vols, anat = {}, {}
    for ph in PHASE_ORDER:
        ct = load_nii(DATA_DIR / rows.loc[ph, 'ct_path'])
        tumor, liver = load_nii(DATA_DIR / rows.loc[ph, 'mask_path']), load_nii(DATA_DIR / rows.loc[ph, 'liver_mask_path'])
        ratio, n_liver, width = density_ratio(liver)
        tz = extent(tumor > 0, 2)
        anat[ph] = {'patient_id': pid, 'class': CLS[pid], 'phase': ph, 'n_slices': int(ct.shape[2]), 'liver_slices': n_liver,
                    'liver_width_px': width, 'ratio': ratio, 'tumour_slices': tz[1] - tz[0] + 1 if tz[0] >= 0 else 0}
        win = None                                                  # E10: in-plane window around liver and tumour (exact)
        if tumor.shape == ct.shape and liver.shape == ct.shape:     # masks on another grid: no window (as E10)
            both = (tumor > 0) | (liver > 0)
            ii, jj = np.where(both.any(axis=(1, 2)))[0], np.where(both.any(axis=(0, 2)))[0]
            if ii.size and jj.size:
                win = (slice(int(ii.min()), int(ii.max()) + 1), slice(int(jj.min()), int(jj.max()) + 1))
            del both
        vols[ph] = tuple(np.ascontiguousarray(a[win]) for a in (ct, tumor, liver)) if win else (ct, tumor, liver)
        del ct, tumor, liver
    pv_ratio = anat['C2']['ratio']
    mct = {}
    for ph in PHASE_ORDER:                                          # E14a rule (00 log section 60)
        own = anat[ph]['ratio']
        ratio, source = (own, 'own') if np.isfinite(own) else (pv_ratio, 'portal_venous') if np.isfinite(pv_ratio) else (np.nan, 'none')
        factor = R_MCT / ratio if np.isfinite(ratio) else 1.0
        anat[ph].update({'mct_factor': float(factor), 'mct_factor_source': source})
        if factor < 1:
            mct[ph] = vols[ph][0].shape[2] * factor
    plan = {'control': {}, 'mct': {ph: (n, False) for ph, n in mct.items()}, 'mct_smooth': {ph: (n, True) for ph, n in mct.items()}}
    if CLS[pid] in ('HCC', 'ICC'):
        n_inject = INJECT_RATIO * vols['C2'][0].shape[2]
        plan['inject_slab'] = {'P': (n_inject, False)}
        plan['inject_slab_smooth'] = {'P': (n_inject, True)}
    arrays = {}
    for name, changes in plan.items():                              # one condition at a time (memory)
        cts, tms, ns = [], [], []
        for ph in PHASE_ORDER:
            v = thicken(vols[ph], *changes[ph]) if ph in changes else vols[ph]
            if name == 'mct':
                anat[ph].update({'mct_n_after': int(v[0].shape[2]), 'mct_ratio_after': density_ratio(v[2])[0],
                                 'mct_tumour_mask_lost': bool(anat[ph]['tumour_slices'] > 0 and not (v[1] > 0).any())})
            if name == 'inject_slab' and ph == 'P':
                anat[ph]['inject_n_after'] = int(v[0].shape[2])
            ct4, tm4 = week2_crop(*v)
            cts.append(ct4); tms.append(tm4); ns.append(v[0].shape[2])
            del v
        arrays[f'{name}_ct'], arrays[f'{name}_tm'], arrays[f'{name}_nslices'] = np.stack(cts), np.stack(tms), np.array(ns)
    np.savez_compressed(WORK / f'{pid}.npz', **arrays)
    return pid, [anat[ph] for ph in PHASE_ORDER], list(plan)

def ram_text():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f' | RAM {vm.used / 1e9:.0f}/{vm.total / 1e9:.0f} GB'
    except Exception:
        return ''

def safe_process(pid):                                              # a data problem in one patient is recorded, not fatal
    try:
        return process_patient(pid) + (None,)
    except Exception as error:
        return pid, [], [], f'{type(error).__name__}: {error}'

t0 = time.time()
ANAT, CONDS, FAILED = [], {}, {}
N_PROC = max(1, min(6, N_CPU - 2)) if IN_COLAB else 1
def collect(results):
    for n, (pid, anat, conds, error) in enumerate(results, start=1):
        if error:
            FAILED[pid] = error; print(f'  {pid}: {error}', flush=True)
        else:
            ANAT.extend(anat); CONDS[pid] = conds
        if n % 25 == 0 or n == len(ALL_IDS):
            print(f'  patients processed {n}/{len(ALL_IDS)} ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)
done = False
if N_PROC > 1:
    try:                                                            # separate processes: resampling does not wait for one another
        collect(joblib.Parallel(n_jobs=N_PROC, return_as='generator')(joblib.delayed(safe_process)(p) for p in ALL_IDS))
        done = True
    except Exception as error:                                     # only infrastructure problems end up here
        print(f'  parallel processing failed ({error!r:.120}); continuing with threads')
        ANAT.clear(); CONDS.clear(); FAILED.clear()
if not done:
    with ThreadPoolExecutor(N_IO) as pool:
        collect(pool.map(safe_process, ALL_IDS))
if FAILED:
    pd.Series(FAILED).to_csv(OUT_DIR / 'E14a_failed_patients.csv')
    raise RuntimeError(f'STOP — {len(FAILED)} patients could not be processed (listed in E14a_failed_patients.csv): {dict(list(FAILED.items())[:5])}')
ANAT = pd.DataFrame(ANAT)
ANAT.to_csv(OUT_DIR / 'E14a_scans_per_phase.csv', index=False)

# ---- gate A: the unchanged condition must reproduce the Week 2 processed crops ----
same = 0
for pid in ALL_IDS:
    with np.load(PROCESSED_DIR / f'{pid}.npz') as z, np.load(WORK / f'{pid}.npz') as w:
        same += int(np.array_equal(z['ct'].astype(np.uint8), w['control_ct']))
GATE = {'control_crops_identical_to_processed': f'{same}/{len(ALL_IDS)}', 'R_MCT': R_MCT}
print('Gate A — unchanged crops identical to the Week 2 processed crops:', GATE['control_crops_identical_to_processed'])
if IN_COLAB and same < 0.99 * len(ALL_IDS):
    raise RuntimeError('STOP — the rebuilt crops do not reproduce the Week 2 crops. Send this output.')

In [ ]:
# Cell 4 — Gate 2 (liver slices and widths equal E10b) and the MCT-like slice density achieved, by phase and class
ref = pd.read_csv(INPUTS['e10b_anatomy'][0], dtype={'patient_id': str}).set_index(['patient_id', 'phase'])
an = ANAT.set_index(['patient_id', 'phase'])
mismatch = [f'{p}/{ph}' for (p, ph) in an.index
            if int(ref.loc[(p, ph), 'liver_slices']) != int(an.loc[(p, ph), 'liver_slices'])
            or int(ref.loc[(p, ph), 'n_slices']) != int(an.loc[(p, ph), 'n_slices'])
            or (int(ref.loc[(p, ph), 'liver_slices']) > 0 and int(ref.loc[(p, ph), 'liver_width_px']) != int(an.loc[(p, ph), 'liver_width_px']))]
GATE['liver_slices_and_widths_equal_E10b'] = 'yes' if not mismatch else f'no: {mismatch[:10]}'
print('Gate 2 — slice counts, liver slices and widths identical to E10b:', GATE['liver_slices_and_widths_equal_E10b'])
if IN_COLAB and mismatch:
    raise RuntimeError('STOP — gate 2 failed. Send this output.')
a = ANAT.copy()
a['thickened'] = a['mct_factor'] < 1
MCT_TABLE = a.groupby(['phase', 'class']).agg(scans=('patient_id', 'size'), thickened=('thickened', 'sum'),
                                              median_factor=('mct_factor', 'median'), median_slices_before=('n_slices', 'median'),
                                              median_slices_after=('mct_n_after', 'median'), median_ratio_before=('ratio', 'median'),
                                              median_ratio_after=('mct_ratio_after', 'median'),
                                              tumour_masks_lost=('mct_tumour_mask_lost', 'sum')).reset_index()
SPREAD_AFTER = {ph: float(g['median_ratio_after'].max() / g['median_ratio_after'].min()) for ph, g in MCT_TABLE.groupby('phase')}
MCT_LIKE_OK = all(s <= 1.25 for s in SPREAD_AFTER.values())
SOURCES = a['mct_factor_source'].value_counts().to_dict()
p_rows = a[a['phase'].eq('P') & a['class'].isin(['HCC', 'ICC'])].set_index('patient_id')
INJECT_CHANGED = sorted(p_rows.index[p_rows['inject_n_after'] < p_rows['n_slices']])
print('MCT-like slice density by phase and class:\n', MCT_TABLE.round(3).to_string(index=False))
print('Class medians of the ratio after slab averaging, max/min by phase:', {k: round(v, 3) for k, v in SPREAD_AFTER.items()},
      '| within 1.25-fold in every phase:', MCT_LIKE_OK, '| factor sources (scans):', SOURCES)
print('Thick non-contrast reconstruction (inject_slab) changed the non-contrast scan of', len(INJECT_CHANGED), 'of', len(p_rows),
      'HCC/ICC patients (the others already had no more slices than 0.29 × portal venous)')
MCT_TABLE.to_csv(OUT_DIR / 'E14a_mct_like_density_by_class.csv', index=False)

In [ ]:
# Cell 5 — Frozen phase-dropout CNN on every condition (own fold model for development patients, ensemble for held-out); gate B
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

D_MODELS = [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)]

@torch.no_grad()
def cnn_predict(pid, ct4):
    x = torch.from_numpy(zscore4(ct4.astype(np.float32))).unsqueeze(0).to(DEVICE)
    models = [D_MODELS[FOLD_OF[pid]]] if pid in FOLD_OF else D_MODELS
    return np.mean([torch.softmax(m(x).float(), dim=1).cpu().numpy()[0] for m in models], axis=0)

rows = []
for pid in ALL_IDS:
    with np.load(WORK / f'{pid}.npz') as w:
        for cond in CONDS[pid]:
            rows.append({'model': 'CNN', 'patient_id': pid, 'condition': cond, **dict(zip(P3, cnn_predict(pid, w[f'{cond}_ct'])))})
PRED = pd.DataFrame(rows)
saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
saved = saved[saved['pattern'] == 'complete']
ref = pd.concat([saved[saved['split'] == 'oof'].set_index('patient_id').loc[dev_ids, P3],
                 saved[saved['split'] == 'test'].set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'CNN') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['cnn_control_vs_E3'] = float(np.abs(ours.to_numpy() - ref.loc[ALL_IDS].to_numpy()).max())
print('Gate B — unchanged CNN predictions vs saved E3 (max difference, must be < 0.02):', GATE['cnn_control_vs_E3'])
if IN_COLAB and GATE['cnn_control_vs_E3'] > 0.02:
    raise RuntimeError('STOP — the frozen CNN does not reproduce E3. Send this output.')

In [ ]:
# Cell 6 — Frozen E2 radiomics model on every condition (Week 5 extraction settings, unchanged); gate C (reported)
RADIOMICS_PARAMS = {                          # Week 5, unchanged
    'setting': {'binWidth': 5, 'label': 1, 'force2D': False, 'normalize': False, 'resampledPixelSpacing': None},
    'imageType': {'Original': {}},
    'featureClass': {'firstorder': None, 'shape': None, 'glcm': None, 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None},
}
def radiomics_patient(pid, conds, work_dir):
    extractor = featureextractor.RadiomicsFeatureExtractor(RADIOMICS_PARAMS)
    out = {}
    with np.load(work_dir / f'{pid}.npz') as w:
        for cond in conds:
            ct4, tm4 = w[f'{cond}_ct'], w[f'{cond}_tm']
            feats = {}
            for i, phase in enumerate(PHASE_ORDER):
                try:
                    result = extractor.execute(sitk.GetImageFromArray(ct4[i].astype(np.float32)), sitk.GetImageFromArray(tm4[i].astype(np.uint8)))
                except Exception:
                    continue
                for key, value in result.items():
                    if not key.startswith('diagnostics_'):
                        try:
                            feats[f'{phase}_{key}'] = float(value)
                        except (TypeError, ValueError):
                            pass
            out[cond] = feats
    return pid, out

t0 = time.time()
jobs = [(pid, CONDS[pid], WORK) for pid in ALL_IDS]
res = None
if N_JOBS > 1:
    try:
        res = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(radiomics_patient)(*job) for job in jobs)
    except Exception as error:
        print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
if res is None:
    res = [radiomics_patient(*job) for job in jobs]
RADF = pd.DataFrame([{'patient_id': pid, 'condition': cond, **feats} for pid, d in res for cond, feats in d.items()])
RADF.to_csv(OUT_DIR / 'E14a_radiomics_features.csv.gz', index=False, compression='gzip')
print(f'Radiomics extracted: {RADF.shape} ({(time.time() - t0) / 60:.1f} min)')

rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
POOL = rad_models[0]['feature_pool']
def rad_predict(pid, row):
    X = row.reindex(POOL).to_numpy(dtype=np.float64)[None, :]
    models = [rad_models[FOLD_OF[pid]]] if pid in FOLD_OF else rad_models
    return np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X[:, m['sel_idx']].astype(np.float32))))[0] for m in models], axis=0)
rows = []
for _, r in RADF.iterrows():
    rows.append({'model': 'radiomics', 'patient_id': r['patient_id'], 'condition': r['condition'], **dict(zip(P3, rad_predict(r['patient_id'], r)))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)

ctrl = RADF[RADF['condition'] == 'control'].set_index('patient_id').loc[ALL_IDS]
ref_feat = internal.set_index('patient_id').loc[ALL_IDS, POOL]
rel = (ctrl.reindex(columns=POOL).astype(float) - ref_feat.astype(float)).abs() / (ref_feat.astype(float).abs() + 1e-6)
ref_rad = pd.concat([pd.read_csv(INPUTS['e2_radiomics_oof'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids, P3],
                     pd.read_csv(INPUTS['e2_radiomics_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'radiomics') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['radiomics_features_vs_week5_max_relative_difference'] = float(np.nanmax(rel.to_numpy()))
GATE['radiomics_control_vs_E2'] = float(np.abs(ours.to_numpy() - ref_rad.loc[ALL_IDS].to_numpy()).max())
print('Gate C — unchanged radiomics: features vs Week 5 (max relative difference):', GATE['radiomics_features_vs_week5_max_relative_difference'],
      '| predictions vs saved E2 (max difference):', GATE['radiomics_control_vs_E2'])
if GATE['radiomics_control_vs_E2'] > 0.02:
    print('NOTE — the re-extracted features do not reproduce Week 5 exactly; every radiomics comparison below is between conditions '
          'extracted here in the same way, so the shifts remain valid, but this is reported.')

In [ ]:
# Cell 7 — Frozen CNN + radiomics meta-learner (the reference fusion of E8b, refitted on the saved out-of-fold inputs); gate D
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
d_saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_saved.query("pattern == 'complete' and split == 'oof'").set_index('patient_id').loc[dev_ids, P3]
oof_meta[W4] = d_oof.loc[oof_meta['patient_id']].to_numpy()

def prepare(X):                               # E8b, unchanged
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
FITTED = []
for f in range(5):                            # E8b meta_folds, unchanged
    tr = oof_meta['fold'].astype(int).ne(f).to_numpy()
    X = prepare(oof_meta.loc[tr, W4 + W5].to_numpy())
    imp = SimpleImputer(strategy='median').fit(X)
    sc = StandardScaler().fit(imp.transform(X))
    m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof_meta.loc[tr, 'label'].astype(int))
    FITTED.append((imp, sc, m))
cnn = PRED[PRED['model'] == 'CNN'].set_index(['patient_id', 'condition'])[P3]
rad = PRED[PRED['model'] == 'radiomics'].set_index(['patient_id', 'condition'])[P3]
rows = []
for (pid, cond) in cnn.index:
    if (pid, cond) not in rad.index:
        continue
    x = prepare(np.concatenate([cnn.loc[(pid, cond)].to_numpy(), rad.loc[(pid, cond)].to_numpy()])[None, :])
    fitted = [FITTED[FOLD_OF[pid]]] if pid in FOLD_OF else FITTED
    p = np.mean([mm.predict_proba(s.transform(i.transform(x)))[0] for i, s, mm in fitted], axis=0)
    rows.append({'model': 'CNN + radiomics', 'patient_id': pid, 'condition': cond, **dict(zip(P3, p))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)
PRED.to_csv(OUT_DIR / 'E14a_predictions.csv', index=False)
e8b = pd.read_csv(INPUTS['e8b_predictions'][0], dtype={'patient_id': str})
e8b = e8b[e8b['model'] == 'cnn_plus_radiomics: [ref] phase_dropout_CNN + radiomics_noshape']
ref_fus = pd.concat([e8b[e8b['cohort'] == 'internal_oof'].set_index('patient_id').loc[dev_ids, P3],
                     e8b[e8b['cohort'] == 'internal_test'].set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'CNN + radiomics') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['fusion_control_vs_E8b'] = float(np.abs(ours.to_numpy() - ref_fus.loc[ALL_IDS].to_numpy()).max())
print('Gate D — unchanged CNN + radiomics vs the E8b reference fusion (max difference):', GATE['fusion_control_vs_E8b'])
if IN_COLAB and GATE['fusion_control_vs_E8b'] > 0.02:
    raise RuntimeError('STOP — the frozen CNN + radiomics does not reproduce E8b. Send this output.')

In [ ]:
# Cell 8 — OA (MCT-like acquisition; primary: held-out HCC versus ICC, CNN + radiomics), OC (thick non-contrast), predicted classes
from scipy.stats import norm
MODELS = ['CNN', 'radiomics', 'CNN + radiomics']
# saved results (fixed in the plan): held-out HCC-versus-ICC AUC (E3 / E2 / E8b) and MCT-LTDiag primary AUC (E6/E8.0, 203 patients)
INTERNAL_REF = {'CNN': 0.961842, 'radiomics': 0.863158, 'CNN + radiomics': 0.947368}
EXTERNAL_REF = {'CNN': 0.527573, 'radiomics': 0.654854, 'CNN + radiomics': 0.578252}
MIDPOINT = 0.763                              # (0.947 + 0.578) / 2, fixed in the plan
OA_CONDS = ['mct', 'mct_smooth']
P = PRED.set_index(['model', 'patient_id', 'condition'])[P3]

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))
def compute_midrank(values):                  # DeLong (E4d/E6, unchanged)
    order = np.argsort(values); sorted_values = values[order]; n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64); index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1; index = end
    output = np.empty(n_values, dtype=np.float64); output[order] = midranks
    return output
def delong_components(y, s):
    pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    return tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m
def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0, auc - 1.96 * se)), float(min(1, auc + 1.96 * se))
def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1); a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))
def probs(model, cond, ids):
    return np.array([P.loc[(model, p, cond)].to_numpy() for p in ids])
def hcc_score(p):
    return p[:, 0] / np.clip(p[:, 0] + p[:, 1], 1e-12, None)

COHORTS = {'held-out': test_ids, 'out-of-fold': dev_ids}
TWO = {c: [p for p in ids if CLS[p] in ('HCC', 'ICC')] for c, ids in COHORTS.items()}
Y2 = {c: np.array([int(CLS[p] == 'HCC') for p in ids]) for c, ids in TWO.items()}
Y3 = {c: np.array([LABEL[p] for p in ids]) for c, ids in COHORTS.items()}
rng = np.random.default_rng(SEED)             # one set of resamples per cohort, shared by every comparison
BOOT2 = {c: [b for b in (rng.integers(0, len(y), len(y)) for _ in range(N_BOOT)) if 0 < y[b].sum() < len(b)] for c, y in Y2.items()}
BOOT3 = {c: [b for b in (rng.integers(0, len(y), len(y)) for _ in range(N_BOOT)) if len(np.unique(y[b])) == 3] for c, y in Y3.items()}

# ---- OA: HCC versus ICC, and three-class macro-AUC, each condition against unchanged inputs (paired) ----
oa, oa3 = [], []
for model in MODELS:
    for cohort in COHORTS:
        ids2, y2 = TWO[cohort], Y2[cohort]
        s0 = hcc_score(probs(model, 'control', ids2))
        auc0 = delong_ci(y2, s0)
        oa.append({'model': model, 'cohort': cohort, 'condition': 'control', 'n': len(ids2), 'auc': auc0[0], 'delong_low': auc0[1],
                   'delong_high': auc0[2]})
        p3_0 = probs(model, 'control', COHORTS[cohort])
        oa3.append({'model': model, 'cohort': cohort, 'condition': 'control', 'n': len(COHORTS[cohort]), 'macro_auc': macro_auc(Y3[cohort], p3_0)})
        for cond in OA_CONDS:
            s1 = hcc_score(probs(model, cond, ids2))
            auc1 = delong_ci(y2, s1)
            d = np.array([roc_auc_score(y2[b], s1[b]) - roc_auc_score(y2[b], s0[b]) for b in BOOT2[cohort]])
            lo, hi = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
            row = {'model': model, 'cohort': cohort, 'condition': cond, 'n': len(ids2), 'auc': auc1[0], 'delong_low': auc1[1],
                   'delong_high': auc1[2], 'difference_vs_unchanged': auc1[0] - auc0[0], 'boot_low': lo, 'boot_high': hi,
                   'delong_paired_p': delong_paired_p(y2, s1, s0)}
            if cohort == 'held-out':
                row['share_of_external_loss_reproduced'] = (auc0[0] - auc1[0]) / (auc0[0] - EXTERNAL_REF[model])
                row['plan_rule_reading'] = ('reproduces most of the external loss' if hi < 0 and auc1[0] < (INTERNAL_REF[model] + EXTERNAL_REF[model]) / 2
                                            else 'explains part of the external loss' if hi < 0 else 'does not explain the external loss')
            oa.append(row)
            p3_1 = probs(model, cond, COHORTS[cohort])
            d3 = np.array([macro_auc(Y3[cohort][b], p3_1[b]) - macro_auc(Y3[cohort][b], p3_0[b]) for b in BOOT3[cohort]])
            oa3.append({'model': model, 'cohort': cohort, 'condition': cond, 'n': len(COHORTS[cohort]), 'macro_auc': macro_auc(Y3[cohort], p3_1),
                        'difference_vs_unchanged': macro_auc(Y3[cohort], p3_1) - macro_auc(Y3[cohort], p3_0),
                        'boot_low': float(np.percentile(d3, 2.5)), 'boot_high': float(np.percentile(d3, 97.5))})
OA, OA3 = pd.DataFrame(oa), pd.DataFrame(oa3)
prim = OA[(OA['model'] == 'CNN + radiomics') & (OA['cohort'] == 'held-out') & (OA['condition'] == 'mct')].iloc[0]
OA_STATEMENT = ('MCT-like acquisition alone reproduces most of the external loss' if prim['boot_high'] < 0 and prim['auc'] < MIDPOINT else
                'MCT-like acquisition explains part of the external loss' if prim['boot_high'] < 0 else
                'MCT-like acquisition does not explain the external loss')

# ---- predicted classes per true class (held-out 56 and all 278) ----
pc = []
for model in MODELS:
    for cond in ['control'] + OA_CONDS:
        for cohort, ids in [('held-out', test_ids), ('all 278', ALL_IDS)]:
            pred = probs(model, cond, ids).argmax(1)
            for true in CLASSES:
                sel = np.array([CLS[p] == true for p in ids])
                pc.append({'model': model, 'condition': cond, 'cohort': cohort, 'true_class': true, 'n': int(sel.sum()),
                           **{f'called_{c}': int(((pred == k) & sel).sum()) for k, c in enumerate(CLASSES)}})
PRED_CLASSES = pd.DataFrame(pc)

# ---- OC: thick non-contrast reconstruction (E10 rules; injection direction only) ----
def boot_ci(v):
    v = np.asarray(v, dtype=float)
    means = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(N_BOOT)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))
oc = []
for model in MODELS:
    for subset, ids in [('all HCC/ICC', INJECT_CHANGED), ('held-out HCC/ICC', [p for p in INJECT_CHANGED if p in set(test_ids)])]:
        ids = [p for p in ids if all((model, p, c) in P.index for c in ('inject_slab', 'inject_slab_smooth', 'control'))]
        if not ids:
            continue
        s_c, s_s, s_0 = (np.array([P.loc[(model, p, c), 'p_CHCC'] for p in ids]) for c in ('inject_slab', 'inject_slab_smooth', 'control'))
        d = s_c - s_s
        lo, hi = boot_ci(d)
        smooth = float(np.mean(np.abs(s_s - s_0)))
        oc.append({'model': model, 'subset': subset, 'n': len(ids), 'mean_shift_vs_smoothing_control': float(d.mean()), 'ci_low': lo,
                   'ci_high': hi, 'mean_shift_vs_unchanged': float((s_c - s_0).mean()), 'smoothing_control_mean_abs_change': smooth,
                   'smoothing_control_ok': smooth < MIN_SHIFT, 'meets_rule': bool(d.mean() >= MIN_SHIFT and lo > 0)})
OC = pd.DataFrame(oc)
OC_VERDICT = {}
for model in MODELS:
    e = OC[(OC['model'] == model) & (OC['subset'] == 'all HCC/ICC')]
    OC_VERDICT[model] = ('not interpretable (smoothing control moved the score)' if not e['smoothing_control_ok'].all() else
                         'yes: responds to a thick non-contrast reconstruction' if e['meets_rule'].all() else 'not shown')
for name, frame in [('E14a_OA_hcc_vs_icc', OA), ('E14a_OA_macro_auc', OA3), ('E14a_predicted_classes', PRED_CLASSES), ('E14a_OC_thick_noncontrast', OC)]:
    frame.to_csv(OUT_DIR / f'{name}.csv', index=False)
print('OA — HCC versus ICC AUC, each condition against unchanged inputs:\n', OA.round(3).to_string(index=False))
print('OA — three-class macro-AUC:\n', OA3.round(3).to_string(index=False))
print('Predicted classes:\n', PRED_CLASSES.to_string(index=False))
print('OC — thick non-contrast reconstruction:\n', OC.round(3).to_string(index=False))
print('OA statement (pre-specified):', OA_STATEMENT, '| OC verdicts:', json.dumps(OC_VERDICT))

In [ ]:
# Cell 9 — Evidence images (seeded choice: one patient per class; portal-venous crop, unchanged / mct / mct_smooth), summary
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
EVID = OUT_DIR / 'E14a_evidence'
EVID.mkdir(exist_ok=True)
pick = np.random.default_rng(SEED)
chosen = [str(pick.choice(sorted(p for p in ALL_IDS if CLS[p] == c))) for c in CLASSES]
for pid in chosen:
    with np.load(WORK / f'{pid}.npz') as w:
        fig, axes = plt.subplots(2, 3, figsize=(10, 7))
        for col, cond in enumerate(['control', 'mct', 'mct_smooth']):
            ct, tm, ns = w[f'{cond}_ct'][2], w[f'{cond}_tm'][2], int(w[f'{cond}_nslices'][2])
            axes[0, col].imshow(ct[:, :, CROP // 2].T, cmap='gray', vmin=0, vmax=255)
            axes[0, col].set_title(f'{pid} ({CLS[pid]}) — {cond}, axial (scan: {ns} slices)', fontsize=8)
            axes[1, col].imshow(ct[:, CROP // 2, :].T, cmap='gray', vmin=0, vmax=255, aspect=1.0)
            if tm[:, CROP // 2, :].any():
                axes[1, col].contour(tm[:, CROP // 2, :].T, levels=[0.5], colors=['#eb6834'], linewidths=0.8)
            axes[1, col].set_title(f'{cond}, coronal through the 96-voxel crop', fontsize=8)
            for ax in axes[:, col]:
                ax.axis('off')
        fig.tight_layout(); fig.savefig(EVID / f'{pid}_{CLS[pid]}_portal_venous.png', dpi=200); plt.close(fig)
print('Evidence images:', sorted(p.name for p in EVID.glob('*.png')))

summary = {'run_id': RUN_ID, 'notebook': 'R17_E14a_mct_like_acquisition_frozen_models.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes, 'gates': GATE,
           'R_MCT': R_MCT, 'mct_like_within_1.25_fold': MCT_LIKE_OK, 'ratio_spread_after': SPREAD_AFTER, 'factor_sources': SOURCES,
           'inject_changed_patients': len(INJECT_CHANGED), 'OA_statement': OA_STATEMENT, 'OC_verdicts': OC_VERDICT,
           'internal_ref': INTERNAL_REF, 'external_ref': EXTERNAL_REF, 'midpoint': MIDPOINT, 'inject_ratio': INJECT_RATIO, 'min_shift': MIN_SHIFT,
           'protocol': '00 log section 60, E14a (written 23 Oct 2026, before the run); frozen models only; nothing trained or selected'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E14a_{RUN_ID} | R = {R_MCT:.6f} | gates: {GATE}')
print('\n--- MCT-like slice density by phase and class ---'); print(MCT_TABLE.round(3).to_string(index=False))
print('ratio spread after:', {k: round(v, 3) for k, v in SPREAD_AFTER.items()}, '| sources:', SOURCES, '| inject changed:', len(INJECT_CHANGED))
print('\n--- OA HCC versus ICC ---'); print(OA.round(3).to_string(index=False))
print('\n--- OA three-class macro-AUC ---'); print(OA3.round(3).to_string(index=False))
print('\n--- predicted classes ---'); print(PRED_CLASSES.to_string(index=False))
print('\n--- OC thick non-contrast ---'); print(OC.round(3).to_string(index=False))
print('\nOA statement (pre-specified):', OA_STATEMENT)
print('OC verdicts:', json.dumps(OC_VERDICT))
print('==============================================================')

In [ ]:
# Cell 10 — Finish: make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()